# ocr.ipynb

This file loads each historical document from `data/` and parses it using OCR to extract text for subsequent feature extraction and analysis. Run on a GPU runtime in Google Colab (preferably T4 GPU).

In [ ]:
# setup
CUDA_TAG = "cu126"
import glob, shutil, sysconfig, importlib.util
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# set dir
DATA_DIR = "/content/drive/MyDrive/armed-occupation-act/data"
WHEELS = "/content/drive/MyDrive/armed-occupation-act/wheels"

# cache paddle wheel
if not glob.glob(f"{WHEELS}/paddlepaddle_gpu*.whl"):
    !pip download -q --no-deps paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/{CUDA_TAG}/ -d {WHEELS}

# install
if importlib.util.find_spec("paddle") is None:
    !pip -q install --find-links {WHEELS} paddlepaddle-gpu
    !pip -q install paddleocr pymupdf

# remove torch
if importlib.util.find_spec("torch") is not None:
    !pip -q uninstall -y torch torchvision torchaudio
    site = Path(sysconfig.get_paths()["purelib"])
    for pat in ["torch", "torch-*", "torchgen", "functorch"]:   # pip leaves stubs behind
        for p in site.glob(pat):
            shutil.rmtree(p, ignore_errors=True)
print("torch:", importlib.util.find_spec("torch"))   # should be None

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: None


In [ ]:
# verify CUDA
!nvidia-smi
import paddle
print("compiled with CUDA:", paddle.device.is_compiled_with_cuda())
print("device:", paddle.device.get_device())
paddle.utils.run_check()
assert paddle.device.is_compiled_with_cuda() and "gpu" in paddle.device.get_device(), \
    "Paddle is not using the GPU!"

Sat Oct 10 20:27:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


compiled with CUDA: True
device: gpu:0
Running verify PaddlePaddle program ... 


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


In [ ]:
# parallel ocr w checkpoint
import os
os.environ.update({"GLOG_minloglevel": "3", "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"})
import json, logging, warnings, threading
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pymupdf
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from paddleocr import PaddleOCR

BATCH = 8            # pages per gpu call
RENDER_WORKERS = 4   # cpu threads rendering pages
DPI = 300

# paths and resume state
pdf = sorted(Path(DATA_DIR).glob("*.pdf"))[0]
txt_dir = pdf.parent / f"{pdf.stem}_pages"
json_dir = pdf.parent / f"{pdf.stem}_json"
txt_dir.mkdir(exist_ok=True)
json_dir.mkdir(exist_ok=True)
final_txt = pdf.with_suffix(".txt")
final_json = pdf.with_suffix(".json")

n_pages = len(pymupdf.open(pdf))
txt_file = lambda i: txt_dir / f"{i+1:04d}.txt"
json_file = lambda i: json_dir / f"{i+1:04d}.json"
todo = [i for i in range(n_pages) if not (txt_file(i).exists() and json_file(i).exists())]
print(f"{pdf.name}: {n_pages} pages, {n_pages - len(todo)} already done, {len(todo)} to go")

# helpers
def save(path, text):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)                                    # atomic checkpoint

def to_record(i, img, r):
    boxes = r["rec_boxes"] if "rec_boxes" in r else [[*np.min(p, 0).tolist(), *np.max(p, 0).tolist()] for p in r["rec_polys"]]
    h, w = img.shape[:2]
    lines = [{"text": t, "score": round(float(s), 4), "box": np.asarray(b).tolist()}   # box is x0,y0,x1,y1 in pixels
             for t, s, b in zip(r["rec_texts"], r["rec_scores"], boxes)]
    return {"page": i + 1, "width": w, "height": h, "lines": lines}

# page renderer
_tl = threading.local()
def render(i):
    if not hasattr(_tl, "doc"):
        _tl.doc = pymupdf.open(pdf)          # one handle per thread
    pix = _tl.doc[i].get_pixmap(dpi=DPI)
    return np.array(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))

# model
ocr = PaddleOCR(device="gpu",
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False)

# run batches
batches = [todo[k:k + BATCH] for k in range(0, len(todo), BATCH)]
with ThreadPoolExecutor(RENDER_WORKERS) as pool, tqdm(total=len(todo), unit="page", desc=pdf.name) as bar:
    submit = lambda b: [pool.submit(render, i) for i in b]
    pending = submit(batches[0]) if batches else []
    for n, idx in enumerate(batches):
        imgs = [f.result() for f in pending]
        pending = submit(batches[n + 1]) if n + 1 < len(batches) else []   # prefetch while gpu runs
        for i, img, r in zip(idx, imgs, ocr.predict(imgs)):
            save(txt_file(i), "\n".join(r["rec_texts"]))
            save(json_file(i), json.dumps(to_record(i, img, r), ensure_ascii=False))
        bar.update(len(idx))

# merge pages
final_txt.write_text("".join(f"\n\n===== PAGE {i+1} =====\n" + txt_file(i).read_text(encoding="utf-8")
                             for i in range(n_pages)), encoding="utf-8")
final_json.write_text(json.dumps([json.loads(json_file(i).read_text(encoding="utf-8")) for i in range(n_pages)],
                                 ensure_ascii=False), encoding="utf-8")
print("Saved:", final_txt, final_json)

SERIALSET-00507_00_00-009-0039-0000.pdf: 66 pages, 0 already done, 66 to go


SERIALSET-00507_00_00-009-0039-0000.pdf:   0%|          | 0/66 [00:00<?, ?page/s]

Saved: /content/drive/MyDrive/armed-occupation-act/data/SERIALSET-00507_00_00-009-0039-0000.txt /content/drive/MyDrive/armed-occupation-act/data/SERIALSET-00507_00_00-009-0039-0000.json
